In [10]:
!pip install transformers torch

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [12]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day!")
        break

    #  Handle important factual questions manually (for accuracy)
    if "artificial intelligence" in user_input.lower():
        print("Chatbot: Artificial Intelligence is the simulation of human intelligence in machines that can learn, reason, and solve problems.")
        continue

    elif "who created python" in user_input.lower():
        print("Chatbot: Python was created by Guido van Rossum in 1991.")
        continue

    elif "thank" in user_input.lower():
        print("Chatbot: You're welcome! Feel free to ask more questions.")
        continue

    # Transformer response for general conversation
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.7,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", bot_response)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot: Hello! I am your AI assistant. How can I help you today?
You: Hello
Chatbot: Hello! :D
You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence is the simulation of human intelligence in machines that can learn, reason, and solve problems.
You: who created Python?
Chatbot: Python was created by Guido van Rossum in 1991.
You: Thank You
Chatbot: You're welcome! Feel free to ask more questions.
You: exit
Chatbot: Goodbye! Have a great day!
